In [ ]:
#pip install nimare

In [1]:
import nilearn as nl
import numpy as np
import matplotlib
import pandas as pd
import os, requests
import nibabel as nib
import matplotlib.pyplot as plt
import openpyxl as xl
import os
import subprocess

alias_dir = "/Users/silvycollin/Documents/research/narratives/fmriprep/"

from nilearn import datasets, image, masking
import numpy as np

import nimare

try:
    # Newer NiMARE versions
    from nimare.datasets import fetch_neurosynth
except ImportError:
    # Older NiMARE versions
    from nimare.extract import fetch_neurosynth



In [3]:

# ----------------------------
# 1. Load the downloaded NeuroSynth STG and MTG maps
# ----------------------------
ns1_img = image.load_img("gyrus stg_association-test_z_FDR_0.01.nii")
ns2_img = image.load_img("mtg_association-test_z_FDR_0.01.nii")

# Threshold (e.g., z >= 3) and binarize each
ns1_data = ns1_img.get_fdata()
ns2_data = ns2_img.get_fdata()

ns1_bin = (ns1_data >= 3.0).astype(int)
ns2_bin = (ns2_data >= 3.0).astype(int)

# Combine both NeuroSynth masks by logical OR (union of STG and MTG)
ns_combined = ((ns1_bin + ns2_bin) > 0).astype(int)
ns_combined_img = image.new_img_like(ns1_img, ns_combined)
ns_combined_img.to_filename("STG_MTG_ns_mask_z3.nii.gz")
print("Saved combined NeuroSynth STG+MTG mask as STG_MTG_ns_mask_z3.nii.gz")
print("Nonzero voxels:", np.sum(ns_combined))

# ----------------------------
# 2. Load Harvard-Oxford cortical atlas
# ----------------------------
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')
atlas_img = image.load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
labels = atlas.labels

# Select medial and temporal regions
target_regions = [
    "Middle Temporal Gyrus, posterior division",
    "Superior Temporal Gyrus, posterior division",
]

# Identify indices for selected regions
indices = [i for i, name in enumerate(labels) if any(r in name for r in target_regions)]
print("Selected HO indices:", indices)
print("Selected labels:", [labels[i] for i in indices])

# Create anatomical mask
anat_mask_data = np.isin(atlas_data, indices).astype(int)
anat_mask_img = image.new_img_like(atlas_img, anat_mask_data)
anat_mask_img.to_filename("posMTGSTG_HO_mask.nii.gz")
print("Saved anatomical medial+temporal mask as Medial_Temporal_HO_mask.nii.gz")

# ----------------------------
# 3. Intersect anatomical & meta-analytic masks
# ----------------------------
refined_data = anat_mask_data * ns_combined
refined_img = image.new_img_like(atlas_img, refined_data)
refined_img.to_filename("posSTG_MTG_refined_mask.nii.gz")
print("Saved final refined STG+MTG mask as STG_MTG_refined_mask.nii.gz")
print("Nonzero voxels:", np.sum(refined_data))

# ----------------------------
# 4. Split into left / right hemispheres
# ----------------------------
affine = refined_img.affine
coords = np.indices(refined_data.shape)
# Compute MNI x-coordinates
x_coords = affine[0, 0] * coords[0] + affine[0, 3]

left_mask = (refined_data > 0) & (x_coords < 0)
right_mask = (refined_data > 0) & (x_coords > 0)

left_img = image.new_img_like(refined_img, left_mask.astype(int))
right_img = image.new_img_like(refined_img, right_mask.astype(int))

left_img.to_filename("posSTG_MTG_refined_mask_L.nii.gz")
right_img.to_filename("posSTG_MTG_refined_mask_R.nii.gz")

print("Saved left hemisphere mask as STG_MTG_refined_mask_L.nii.gz")
print("Saved right hemisphere mask as STG_MTG_refined_mask_R.nii.gz")
print("Left voxels:", np.sum(left_mask), "Right voxels:", np.sum(right_mask))


Saved combined NeuroSynth STG+MTG mask as STG_MTG_ns_mask_z3.nii.gz
Nonzero voxels: 4913


/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_1554/620088945.py:16: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  ns_combined_img = image.new_img_like(ns1_img, ns_combined)


[get_dataset_dir] Dataset found in /Users/silvycollin/nilearn_data/fsl

Selected HO indices: [10, 12]
Selected labels: ['Superior Temporal Gyrus, posterior division', 'Middle Temporal Gyrus, posterior division']
Saved anatomical medial+temporal mask as Medial_Temporal_HO_mask.nii.gz
Saved final refined STG+MTG mask as STG_MTG_refined_mask.nii.gz
Nonzero voxels: 1375
Saved left hemisphere mask as STG_MTG_refined_mask_L.nii.gz
Saved right hemisphere mask as STG_MTG_refined_mask_R.nii.gz
Left voxels: 652 Right voxels: 723


/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_1554/620088945.py:42: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  anat_mask_img = image.new_img_like(atlas_img, anat_mask_data)
/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_1554/620088945.py:50: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  refined_img = image.new_img_like(atlas_img, refined_data)
/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_1554/620088945.py:66: UserWarning: Data array used to create a new image cont